# Session 3: Decision Trees & Random Forests
All exercises in one notebook. Course data downloads automatically, so just run top to bottom. The ANSUR data (last section) needs a separate download, explained there.

In [ ]:
!pip install -q sdv imbalanced-learn

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import tree
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, BaggingClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score,
                             precision_score, recall_score, f1_score)

warnings.filterwarnings("ignore")
np.random.seed(42)
BASE = "https://raw.githubusercontent.com/Part-Institute/MasterClass-FinTech-01/main/session_3/"

## 1. Comedy show
### A) Augmenting with sdv
Only 13 rows, so, as the exercise asks, we add 100 synthetic rows with `sdv` and train on the augmented set.

In [ ]:
from sdv.metadata import Metadata
from sdv.single_table import GaussianCopulaSynthesizer

df = pd.read_csv(BASE + "1.decision_tree_comedy_show/data.csv")

metadata = Metadata.detect_from_dataframe(df, table_name="comedian")
synthesizer = GaussianCopulaSynthesizer(metadata)
synthesizer.fit(df)
synthetic_data = synthesizer.sample(num_rows=100)

df_augmented = pd.concat([df, synthetic_data], ignore_index=True)
print(df.shape, "->", df_augmented.shape)
df_augmented.head()

In [ ]:
data = df_augmented.copy()
data["Nationality"] = data["Nationality"].map({"UK": 0, "USA": 1, "N": 2})
data["Go"] = data["Go"].map({"YES": 1, "NO": 0})
data = data.dropna().astype(int)

features = ["Age", "Experience", "Rank", "Nationality"]
X, y = data[features], data["Go"]

dtree = DecisionTreeClassifier(random_state=42).fit(X, y)
plt.figure(figsize=(14, 7))
tree.plot_tree(dtree, feature_names=features, class_names=["NO", "YES"], filled=True)
plt.title("Decision Tree (gini) on augmented data")
plt.show()

### B) Gini vs entropy
**Gini** scores how mixed a node is: the chance of mislabeling a random sample, `Gini = 1 - sum(p_i^2)`. A pure node scores 0. At each step the tree picks the split that cuts the weighted Gini of the children the most.

In [ ]:
res = {}
for crit in ["gini", "entropy"]:
    m = DecisionTreeClassifier(criterion=crit, random_state=42).fit(X, y)
    res[crit] = m
    print(f"{crit:8s} -> depth: {m.get_depth()}, leaves: {m.get_n_leaves()}")

imp = pd.DataFrame({c: m.feature_importances_ for c, m in res.items()}, index=features).round(3)
display(imp)
imp.plot.bar(figsize=(7, 4), title="Feature importance: gini vs entropy", ylabel="importance", xlabel="feature")
plt.xticks(rotation=0); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, (c, m) in zip(axes, res.items()):
    tree.plot_tree(m, feature_names=features, filled=True, ax=ax)
    ax.set_title(c)
plt.show()

**Entropy** uses `-sum(p_i log2 p_i)` instead. It punishes mixed nodes a bit harder, but in practice both trees split almost the same way: same top splits, similar depth and leaf count, same feature ranking. Only a few deep splits or small importance values move.

**What I noticed:** the two criteria agree so much that picking one is mostly a coin flip. On 100+ rows, the differences are tiny.

## 2. Boston house-prices
Boston was removed from recent scikit-learn, so we read the same dataset from a public CSV (506 rows, target `medv` in $1000s).

In [ ]:
boston = pd.read_csv("https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv")
X = boston.drop(columns="medv")
y = boston["medv"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

### B & C) Random Forest, 100 trees + metrics

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)
pred = rf.predict(X_test)

mse = mean_squared_error(y_test, pred)
metrics = pd.Series({
    "MSE": mse,
    "RMSE": np.sqrt(mse),
    "MAE": mean_absolute_error(y_test, pred),
    "MAPE": mean_absolute_percentage_error(y_test, pred),
    "R2": r2_score(y_test, pred),
}).round(4)
metrics

### D) Actual vs predicted

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, pred, alpha=0.7, label="test samples")
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
plt.plot(lims, lims, "r--", label="perfect prediction")
plt.xlabel("Actual price (x $1000)"); plt.ylabel("Predicted price (x $1000)")
plt.title("Random Forest (100 trees): actual vs predicted")
plt.legend(); plt.show()

### E & F) 50 vs 100 vs 200 trees, RMSE in dollars

In [ ]:
rows = []
for n in [50, 100, 200]:
    m = RandomForestRegressor(n_estimators=n, random_state=42).fit(X_train, y_train)
    p = m.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, p))
    rows.append({"n_estimators": n, "test RMSE (x$1000)": round(rmse, 4),
                 "test RMSE ($)": round(rmse * 1000), "test R2": round(r2_score(y_test, p), 4)})
table = pd.DataFrame(rows)
display(table)

plt.figure(figsize=(6, 4))
bars = plt.bar(table["n_estimators"].astype(str), table["test RMSE ($)"], color=["#8da0cb", "#66c2a5", "#fc8d62"])
plt.bar_label(bars, fmt="$%d")
plt.xlabel("Number of trees"); plt.ylabel("Test RMSE (US dollars)")
plt.title("Boston: RMSE in dollars for different forest sizes")
plt.show()

Prices are in $1000s, so RMSE x 1000 gives dollars: the model is off by roughly $2.8k on average.

 **What I noticed:** more trees didn't help. On this split the gap is tiny and 50 trees even came out slightly better. Past a point, extra trees only cost time.

## 3. Titanic

In [ ]:
titanic = pd.read_csv(BASE + "4.random_forest_titanic/titanic.csv")
titanic = titanic.dropna(subset=["Survived"])
titanic["Sex"] = titanic["Sex"].map({"female": 0, "male": 1})

feats = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]
X_train, X_test, y_train, y_test = train_test_split(titanic[feats], titanic["Survived"], test_size=0.2, random_state=42)

age_median = X_train["Age"].median()
X_train = X_train.fillna({"Age": age_median})
X_test = X_test.fillna({"Age": age_median})
print("median age (train only):", age_median)

rf_t = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)

### A) Feature importance

In [ ]:
fi = pd.Series(rf_t.feature_importances_, index=feats).sort_values(ascending=False)
display(fi.round(3))
fi.sort_values().plot.barh(figsize=(7, 4), color="teal")
plt.xlabel("Feature importance"); plt.ylabel("Feature"); plt.title("Titanic Random Forest: feature importances")
plt.show()

`Fare`, `Sex` and `Age` lead. That fits the history: women and higher-class passengers were saved more often. `Parch` and `SibSp` matter least.

### C) Classification report & confusion matrix

In [ ]:
y_pred = rf_t.predict(X_test)
names = ["Did not survive", "Survived"]
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred, target_names=names))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=names).plot(cmap="Blues")
plt.title("Titanic confusion matrix (test)"); plt.show()

### D) Errors (Survived = positive class)
- **False positive:** model said survived, passenger actually died (about 16).
- **False negative:** model said died, passenger actually survived (about 20).
- Recall is higher for `Did not survive` (about 0.85 vs 0.73): it's the majority class, so the model finds deaths more easily than survivors.

### E) Effect of max_depth

In [ ]:
rows = []
for d in [1, 2, 4, 8]:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    rows.append({"max_depth": d, "actual depth": dt.get_depth(), "n_leaves": dt.get_n_leaves(),
                 "train accuracy": round(dt.score(X_train, y_train), 4)})
display(pd.DataFrame(rows))

dt2 = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_train, y_train)
plt.figure(figsize=(10, 5))
tree.plot_tree(dt2, feature_names=feats, class_names=names, filled=True)
plt.title("Decision tree with max_depth=2"); plt.show()

### F) Kaggle submission
The model is fit on all of train and predicts the original Kaggle `test.csv`. Median age and fare come from train only.

In [ ]:
test_kaggle = pd.read_csv("https://raw.githubusercontent.com/agconti/kaggle-titanic/master/data/test.csv")

full = titanic.copy()
age_med, fare_med = full["Age"].median(), full["Fare"].median()
X_full = full[feats].fillna({"Age": age_med, "Fare": fare_med})
y_full = full["Survived"]

X_sub = test_kaggle.copy()
X_sub["Sex"] = X_sub["Sex"].map({"female": 0, "male": 1})
X_sub = X_sub[feats].fillna({"Age": age_med, "Fare": fare_med})

final_rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_full, y_full)
submission = pd.DataFrame({"PassengerId": test_kaggle["PassengerId"], "Survived": final_rf.predict(X_sub)})
submission.to_csv("submission.csv", index=False)
print(submission.shape); submission.head()

**Kaggle test accuracy: 0.804**

## 4. Give Me Some Credit
### A) Preparation and train/test export
Compact version of the course prep steps: drop the leaking column (`NumberOfDefault`), remove impossible ages, split, fill missing values with train medians, add new features.

In [ ]:
credit = pd.read_csv(BASE + "5.give_me_some_credit/GiveMeSomeCredit.csv")
credit["SEX"] = credit["SEX"].map({"M": 0, "F": 1})
credit = credit.drop(columns=["NumberOfDefault"])
credit = credit[(credit["age"] >= 18) & (credit["age"] <= 100)]

target = "SeriousDlqin"
Xc, yc = credit.drop(columns=[target]), credit[target]
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.30, random_state=42, stratify=yc)
train_df = Xc_tr.assign(**{target: yc_tr}); test_df = Xc_te.assign(**{target: yc_te})

for d in (train_df, test_df):
    d["MonthlyIncome_missing"] = d["MonthlyIncome"].isna().astype(int)
for col in ["MonthlyIncome", "NumberOfDependents"]:
    med = train_df[col].median()
    train_df[col] = train_df[col].fillna(med); test_df[col] = test_df[col].fillna(med)

num_cols = ["age", "RevolvingUtilizationOfUnsecuredLines", "DebtRatio", "MonthlyIncome",
            "NumberOfOpenCreditLinesAndLoans", "NumberOfTimes90DaysLate", "NumberRealEstateLoansOrLines",
            "NumberOfTime30-59DaysPastDueNotWorse", "NumberOfTime60-89DaysPastDueNotWorse", "NumberOfDependents"]
train_df["Is_Outlier"] = False
for col in num_cols:
    q1, q3 = train_df[col].quantile([0.25, 0.75]); iqr = q3 - q1
    train_df["Is_Outlier"] |= (train_df[col] < q1 - 1.5 * iqr) | (train_df[col] > q3 + 1.5 * iqr)

for d in (train_df, test_df):
    d["TotalPastDue"] = (d["NumberOfTime30-59DaysPastDueNotWorse"] + d["NumberOfTime60-89DaysPastDueNotWorse"]
                         + d["NumberOfTimes90DaysLate"])
    d["Has90DaysLate"] = (d["NumberOfTimes90DaysLate"] > 0).astype(int)
    d["WorstDelayStatus"] = np.select([d["NumberOfTimes90DaysLate"] > 0,
                                       d["NumberOfTime60-89DaysPastDueNotWorse"] > 0,
                                       d["NumberOfTime30-59DaysPastDueNotWorse"] > 0], [3, 2, 1], default=0)

train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)
print(train_df.shape, test_df.shape)

### B) Baseline Random Forest + precision / recall / F1

In [ ]:
tr = pd.read_csv("train.csv"); te = pd.read_csv("test.csv")
X_tr, y_tr = tr.drop(columns=[target, "Is_Outlier"]), tr[target]
X_te, y_te = te.drop(columns=[target]), te[target]

rf_c = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
p = rf_c.predict(X_te)
print("Accuracy:", round(accuracy_score(y_te, p), 4), "| majority-class share:", round((y_te == 0).mean(), 4))
print(classification_report(y_te, p, target_names=["No delinquency (0)", "Delinquency (1)"]))
ConfusionMatrixDisplay(confusion_matrix(y_te, p), display_labels=["No delinquency", "Delinquency"]).plot(cmap="Blues")
plt.title("Random Forest baseline: confusion matrix (test)"); plt.show()

**Why accuracy alone misleads:** about 93% of customers don't default, so a model that predicts "fine" for everyone scores ~93% without catching a single defaulter.

Here accuracy is about 94%, but recall for class 1 is only about 0.19: we miss roughly 4 of every 5 defaulters, and for a bank that miss (false negative) is the costly one.

### C) Improving the model
We try balancing (SMOTENC), outlier removal, a different model (HistGradientBoosting) and tuning, alone and combined. Everything is scored on the same test set.

In [ ]:
from imblearn.over_sampling import SMOTENC
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV

cat_cols = ["SEX", "EDUCATION", "MonthlyIncome_missing", "Has90DaysLate", "WorstDelayStatus"]
cat_idx = [list(X_tr.columns).index(c) for c in cat_cols]

clean = tr[~tr["Is_Outlier"]]
X_clean, y_clean = clean.drop(columns=[target, "Is_Outlier"]), clean[target]
print("rows after outlier removal:", len(clean), "of", len(tr), "| delinquent rate:", round(y_clean.mean(), 4), "vs", round(y_tr.mean(), 4))

X_sm, y_sm = SMOTENC(categorical_features=cat_idx, random_state=42).fit_resample(X_tr, y_tr)

results = []
def run(name, model, X, y):
    model.fit(X, y); pr = model.predict(X_te)
    results.append({"model": name, "accuracy": accuracy_score(y_te, pr), "precision(1)": precision_score(y_te, pr),
                    "recall(1)": recall_score(y_te, pr), "f1(1)": f1_score(y_te, pr)})
    return model

rf_new = lambda **k: RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, **k)
run("RF baseline", rf_new(), X_tr, y_tr)
run("RF + remove outliers", rf_new(), X_clean, y_clean)
run("RF + SMOTENC", rf_new(), X_sm, y_sm)
run("RF + class_weight balanced", rf_new(class_weight="balanced_subsample", min_samples_leaf=5), X_tr, y_tr)
run("HistGB default", HistGradientBoostingClassifier(random_state=42), X_tr, y_tr)
run("HistGB + SMOTENC", HistGradientBoostingClassifier(random_state=42), X_sm, y_sm)

In [ ]:
param_dist = {"learning_rate": [0.03, 0.05, 0.1], "max_iter": [100, 200, 300],
              "max_depth": [3, 4, 6, None], "min_samples_leaf": [20, 50, 100], "l2_regularization": [0, 1, 10]}
search = RandomizedSearchCV(HistGradientBoostingClassifier(random_state=42), param_dist, n_iter=10,
                            scoring="accuracy", cv=3, random_state=42, n_jobs=-1)
search.fit(X_tr, y_tr)
best_params = search.best_params_
print("best params:", best_params)

run("HistGB tuned (best CV accuracy)", search.best_estimator_, X_tr, y_tr)
run("HistGB tuned + class_weight balanced",
    HistGradientBoostingClassifier(random_state=42, class_weight="balanced", **best_params), X_tr, y_tr)

res_df = pd.DataFrame(results).round(4).sort_values("accuracy", ascending=False).reset_index(drop=True)
display(res_df)

In [ ]:
best = res_df.iloc[0]
print("Best model by accuracy:", best["model"], "| accuracy =", best["accuracy"])
print("Best hyperparameters:", best_params)

res_df.set_index("model")[["accuracy", "recall(1)", "f1(1)"]].plot.barh(figsize=(9, 5))
plt.xlabel("score (test)"); plt.title("Model comparison on test set"); plt.gca().invert_yaxis(); plt.show()

**Summary:** the best accuracy goes to the tuned HistGradientBoosting (about 93.8%), only slightly above the baseline RF.

Outlier removal with IQR backfired: it throws away about half of train and most defaulters sit in that half (recall nearly 0).

Balancing and `class_weight` lower accuracy but raise defaulter recall a lot (from about 0.19 to about 0.48 or more).

## 5. Women vs Men
ANSUR II comes from Kaggle (`seshadrikolluri/ansur-ii`) or OpenLab. The cell below tries `kagglehub` first; if that fails, upload `ANSUR II MALE Public.csv` and `ANSUR II FEMALE Public.csv` manually.

In [ ]:
import glob

def read_any(path):
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1")

def find_files():
    files = glob.glob("*.csv") + glob.glob("/root/.cache/kagglehub/**/*.csv", recursive=True)
    low = lambda f: os.path.basename(f).lower()
    fem = [f for f in files if "female" in low(f) and "ansur" in low(f)]
    mal = [f for f in files if "male" in low(f) and "female" not in low(f) and "ansur" in low(f)]
    return (mal[0], fem[0]) if mal and fem else None

found = find_files()
if not found:
    try:
        import kagglehub
        kagglehub.dataset_download("seshadrikolluri/ansur-ii")
        found = find_files()
    except Exception as e:
        print("kagglehub failed:", e)
if not found:
    from google.colab import files as colab_files
    print("Pick the two ANSUR files:")
    colab_files.upload()
    found = find_files()

male, female = read_any(found[0]), read_any(found[1])
print(found, male.shape, female.shape)

### B) Preprocessing

In [ ]:
ansur = pd.concat([male, female], ignore_index=True)
ansur.columns = [c.strip().lower() for c in ansur.columns]

drop_cols = ["subjectid", "subjectnumericrace", "dodrace", "ethnicity", "age", "installation", "component",
             "branch", "primarymos", "writingpreference", "date", "subjectsbirthlocation", "heightin", "weightlbs"]
ansur = ansur.drop(columns=[c for c in drop_cols if c in ansur.columns])
ansur = ansur.drop_duplicates()
ansur["gender"] = ansur["gender"].astype(str).str.strip().str.lower().map({"male": 0, "female": 1})

feature_cols = ansur.select_dtypes(include=np.number).columns.drop("gender")
ansur = ansur.dropna(subset=["gender"]).dropna(subset=feature_cols)
print(ansur.shape, "| male:", (ansur.gender == 0).sum(), "female:", (ansur.gender == 1).sum())

### C) Heights of women and men

In [ ]:
height = ansur["stature"] / 10
plt.figure(figsize=(8, 4.5))
plt.hist(height[ansur.gender == 0], bins=40, alpha=0.6, label="Men", color="steelblue")
plt.hist(height[ansur.gender == 1], bins=40, alpha=0.6, label="Women", color="indianred")
plt.xlabel("Height (cm)"); plt.ylabel("Number of people"); plt.title("Height distribution: women vs men (ANSUR II)")
plt.legend(); plt.show()
print(height.groupby(ansur.gender).agg(["mean", "std"]).rename(index={0: "Men", 1: "Women"}).round(1))

**Why are men's heights higher?** Hormones (testosterone) and a longer growth period in boys push the average about 13 cm above girls'.

**Why is the men's curve further right?** The whole distribution is shifted toward larger values, and the spread (std) is slightly bigger, so it's wider too.

💡 **What I noticed:** the two histograms overlap a lot, yet height alone is a weak classifier. The 100+ body measurements together are what make the classes separable.

### D to G) Decision Tree vs Bagging

In [ ]:
X = ansur[feature_cols]; y = ansur["gender"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

single = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
bag = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, random_state=42, n_jobs=-1).fit(X_train, y_train)

for name, m in [("Single Decision Tree", single), ("Bagging (100 trees)", bag)]:
    print(f"{name:22s} train acc = {m.score(X_train, y_train):.4f} | test acc = {m.score(X_test, y_test):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, m) in zip(axes, [("Decision Tree", single), ("Bagging", bag)]):
    ConfusionMatrixDisplay(confusion_matrix(y_test, m.predict(X_test)), display_labels=["Male", "Female"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name}: confusion matrix (test)")
plt.show()

Bagging averages 100 trees, which cuts variance, so test accuracy is usually higher and errors fewer than a single tree (which hits ~100% on train, i.e. overfits). The numbers above show the exact comparison.